# Scoring fine-mapped blood-lipid variants with HepG2 ChromBPNet models

Exploratory. HepG2 (hepatocyte) released ATAC + DNase ChromBPNet models; lipid traits (LDL-C / HDL-C / TG).
Structure follows the README: **question → method → one figure → what surprised me → limitations**.

Order matters: the CAGI5 correctness check runs before we trust any downstream enrichment.

In [ ]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd
from src.config import load_config
from src import (cagi5_repro, ldsc_enrichment, variant_scoring,
                 pip_enrichment, assay_concordance, contributions, plots)
cfg = load_config('../config.yaml')
cfg['traits']

## 1. Correctness check — reproduce SORT1 / LDLR CAGI5 (HepG2 DNase)

Reproduce the paper's Table 4 HepG2 DNase MPRA correlations before doing anything new.
rs12740374 (SORT1, C/EBP-site-creating) is the anchor. If these don't reproduce, stop.

In [ ]:
cagi5 = cagi5_repro.run(cfg)
cagi5  # compare pearson/spearman per locus against paper Table 4

## 2. Cell-type justification — LDSC lipid-heritability enrichment in HepG2 regions

Mirror of the paper's K562/blood step. Enrichment here is what earns the cell-type choice quantitatively.

In [ ]:
ldsc = pd.read_csv('../results/tables/ldsc_enrichment.csv')  # produced by run_all.sh step 2
ldsc

## 3–4. Score fine-mapped lipid variants (both assays) and test PIP concordance

Counterfactual ref-vs-alt: logFC counts + profile JSD, per assay. Then enrichment of high scores across PIP bins.

In [ ]:
enr = pip_enrichment.enrichment(cfg, assay='dnase')
plots.pip_enrichment_figure(enr, '../results/figures/pip_enrichment.png')
enr

## 5. NEW — do DNase and ATAC agree on counterfactual variant effects?

Reference accessibility agrees after bias correction (paper). Variant-effect agreement is the unreported question.
Highlight the disagreers — profile-shape vs coverage, or instability.

In [ ]:
m = assay_concordance.merge_assays(cfg)
print(assay_concordance.summarize(cfg, m))
plots.assay_scatter(m, '../results/figures/atac_vs_dnase_variant_scores.png')

## 6. Mechanistic coherence — DeepSHAP + HepG2 motif check at top hits

Do the top variant effects disrupt the HepG2 lexicon (HNF4A, FOXA, CEBP, HNF1A; paper Fig. 5a)?
Right motifs = coherence, not just correlation.

In [ ]:
hits = contributions.top_hits(cfg)
hits[['variant_id','trait','logfc_counts','jsd_profile']].head(20)
# contributions.deepshap(cfg, hits); contributions.motif_coherence(cfg)

## What surprised me

*(one or two honest sentences after running)*

## Limitations

- HepG2 is a hepatoblastoma line, not primary hepatocytes.
- Enrichment is concordance with fine-mapping, not causality — LDSC + motif coherence are the guards.
- One cell type, released models, no retraining — an observation, not a benchmark.